# NuCLS · Experimento v2 en GPU
**Andrés Duarte — Deep Learning**

Ejecución de softmaxCNN-BN adaptada, DenseNet121 y ResNet50 desde cero. Se mantienen la partición por pacientes y centros, la búsqueda de hiperparámetros y la regla de planificación de corridas fijadas en el protocolo.

## Preparación
1. Subir `NuCLS_V2_DATOS.zip` a la raíz de **Mi unidad** de Google Drive.
2. En Colab seleccionar **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**.
3. Ejecutar las celdas en orden. Las imágenes se leen desde el disco temporal de Colab; los checkpoints y resultados se guardan en Drive.

La ejecución incluye 12 ajustes de búsqueda, 15 del piloto y al menos 15 finales. Puede requerir varias horas o sesiones; no se estima duración sin medir la GPU. Se guardan checkpoints por época. La planificación se detiene si requiere más de 50 corridas finales por modelo, sin evaluar la reserva.

Este cuaderno se entrega sin salidas de entrenamiento: los resultados deben proceder de su ejecución real. El entrenamiento no sustituye el cierre de la revisión bibliográfica ni el análisis agrupado de los seis pacientes de reserva.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, importlib.util
for module, package in [('numpy','numpy'),('scipy','scipy'),('PIL','Pillow')]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install',package],check=True)
import torch, torchvision, numpy, scipy
if not torch.cuda.is_available():
    raise RuntimeError('Seleccionar GPU en el menú Entorno de ejecución antes de continuar.')
print('GPU:',torch.cuda.get_device_name(0))
print('PyTorch:',torch.__version__,'Torchvision:',torchvision.__version__)
print('NumPy:',numpy.__version__,'SciPy:',scipy.__version__)


## Datos y código fijados
Se verifica el manifiesto y la integridad binaria de los archivos durante la extracción, sin calcular características ni predicciones de la reserva. Los archivos originales en Drive permanecen intactos.

In [ ]:
import hashlib, csv, zipfile, urllib.request, shutil
COMMIT='6d672bcb13a33536f9e1c48d002c6f7c513d969e'
EXPECTED='623adf36520751addc324e44cd1b89d95e47842e53cda494914c0248a8be86d4'
ZIP_DRIVE=Path('/content/drive/MyDrive/NuCLS_V2_DATOS.zip')
DATOS=Path('/content/nucls_v2_datos')
CODIGO=Path('/content/nucls_v2_codigo/modelos')
SALIDA=Path('/content/drive/MyDrive/NuCLS_V2_RESULTADOS_6d672bcb')
if not ZIP_DRIVE.is_file():
    raise FileNotFoundError('Subir NuCLS_V2_DATOS.zip a la raíz de Mi unidad.')
local_zip=Path('/content/NuCLS_V2_DATOS.zip')
shutil.copyfile(ZIP_DRIVE, local_zip)
DATOS.mkdir(exist_ok=True); CODIGO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    raw=z.read('manifiesto.csv')
    if hashlib.sha256(raw).hexdigest()!=EXPECTED:
        raise ValueError('El manifiesto no corresponde al conjunto v2 verificado.')
    rows=list(csv.DictReader(raw.decode().splitlines()))
    if len(rows)!=17557: raise ValueError('Número de recortes inesperado.')
    for i,row in enumerate(rows,1):
        target=(DATOS/row['archivo']).resolve()
        if not target.is_relative_to(DATOS.resolve()): raise ValueError('Ruta no válida.')
        data=z.read(row['archivo'])
        if hashlib.sha256(data).hexdigest()!=row['sha_archivo']:
            raise ValueError('Archivo diferente: '+row['archivo'])
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(data)
        if i%2000==0: print('Recortes verificados:',i,flush=True)
    (DATOS/'manifiesto.csv').write_bytes(raw)
for name in ['candidatos_v2.py','core_v2.py','experimento_v2.py']:
    url=f'https://raw.githubusercontent.com/aduartech81/NuCLS_DeepLearning/{COMMIT}/modelos/{name}'
    with urllib.request.urlopen(url,timeout=60) as response: data=response.read()
    compile(data,name,'exec');(CODIGO/name).write_bytes(data)
print('Recortes disponibles:',len(rows))
print('Resultados y checkpoints:',SALIDA)


## Entrenamiento y evaluación
La selección se realiza exclusivamente con validación. Antes de evaluar la reserva se fijan el modelo de referencia, las semillas y las huellas de todos los checkpoints finales.

Si Colab se desconecta, se puede volver a ejecutar el cuaderno con la misma carpeta de salida. El motor comprueba configuración, código y entorno antes de reanudar; si detecta cambios, se detiene para evitar mezclar ejecuciones. No debe borrarse la carpeta de resultados para resolver ese mensaje. Drive necesita espacio para varios checkpoints por corrida.

In [ ]:
subprocess.run([
    sys.executable,str(CODIGO/'experimento_v2.py'),
    '--datos',str(DATOS),'--salida',str(SALIDA),'--device','cuda'
],check=True)


## Evidencia de ejecución
El ZIP contiene métricas, predicciones y registros; los pesos se conservan por separado en Drive. El estado distingue un experimento completado de una interrupción por el límite de corridas. Una ejecución completa requiere posteriormente interpretar los resultados e incorporarlos al informe y al dashboard.

In [ ]:
import json
estado=SALIDA/'estado.json'
plan=SALIDA/'plan_corridas.json'
if estado.exists():
    print(json.dumps(json.loads(estado.read_text()),indent=2,ensure_ascii=False))
elif plan.exists():
    print(json.dumps(json.loads(plan.read_text()),indent=2,ensure_ascii=False))
else:
    print('Ejecución incompleta. Revisar el mensaje anterior; los checkpoints permanecen en Drive.')
for path in sorted(SALIDA.parent.glob(SALIDA.name+'*.zip')):
    print('ZIP de resultados:',path)
